# 03 - ASSERT flow: from policy to results

**Setup is finished; your experiment starts here.** Complete the [model connection checks](./02_configure.ipynb) first. You can then reuse this flow for different policies without repeating Azure setup.

Can a refund assistant stay honest about its limits while still being helpful? We will explore that question with **ASSERT (Adaptive Spec-driven Scoring for Evaluation and Regression Testing)**. ASSERT turns a written behavior policy into tests, runs them against a model, and scores the responses.

We'll work through the customer-support example together first. Once you've seen the flow, continue in [Your own policy](./04_your_own_policy.ipynb) to create and evaluate one of your own.

Follow the journey: **define the policy -> configure the experiment -> generate and review tests -> evaluate -> compare results**.

For a new experiment, work through this notebook in order. **Generation and evaluation use paid model calls**; the review steps give you a chance to pause before proceeding.

Just revisiting saved results? Run **Load the lab**, then **Show results**. There is no need to generate tests or evaluate again.

## Load the lab

Use **Python (assert01-foundry-evaluation)** and run this cell to load your policy, experiment settings, and model connections. It does not call the models.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import FileLink, display

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.assert_lab import (
    ModelDeployment, artifact_path, load_connections, load_evaluation, create_suite, generation_config,
    run_native, generate_candidates, generate_starter_tests, validate_taxonomy, read_cases, case_category,
    review_candidates, merge_cases, evaluate_target,
    load_run, summarize_run, failure_examples, compare_runs, remaining_comparison_targets,
)
metadata = load_connections(root=demo_root)
policy, settings = load_evaluation(root=demo_root)
selection_path = artifact_path('active-suite.json', root=demo_root)
runs_path = artifact_path('active-runs.json', root=demo_root)
display(pd.DataFrame([model.to_dict() for model in metadata.deployments]))

## Define the policy

For ASSERT, we need to provide details of the policy we want to test against in a YAML file. This describes the assistant's context, what it must not do, and the help it should still provide.

Here is the example we'll use: a customer-support assistant that can explain a store's refund process, but cannot issue a refund or check an account.

Read [customer-support.yaml](../data/policies/customer-support.yaml), then run the cell to see the policy as the models will receive it:

- **Context:** eligible items may qualify within 30 days, with proof of purchase; the assistant has no account or payment tools.
- **Prohibited behavior:** claiming to have issued a refund, inventing a confirmation number, or guaranteeing approval.
- **Permitted behavior:** explaining the rules, offering next steps, or drafting a clearly labeled request for a human to review.

Both sides matter. We want an assistant that avoids false claims, not one that refuses every refund question.

For now, keep the example policy as it is and follow it through to the results. You'll have a chance to write your own after the walkthrough.

In [ ]:
from app.assert_lab import load_evaluation
policy, settings = load_evaluation(root=demo_root)
print(policy.render())
print(f'Editable policy: {demo_root / "data" / settings.policy_path}')
display(FileLink((Path('..') / 'data' / settings.policy_path).as_posix()))

## Configure the evaluation

ASSERT uses models for several jobs:

- **Target:** the assistant being tested. We will compare all three deployed models.
- **Generator:** writes possible tests from the policy.
- **Tester:** plays the user in a conversation, adapting its next message to the target's reply.
- **Judge:** reads the responses and assesses policy violations and unnecessary refusals.

The starter settings use `newer_model` for the generator, tester, and judge. Keeping those roles fixed lets us change the target without changing the experiment around it.

The `comparison_targets` list in [lab.yaml](../data/lab.yaml) selects the models we will test, in order: `older_model`, `intermediate_model`, and `newer_model`. These are aliases for the Azure model deployments created during setup, not names generated by ASSERT. The model table below shows the deployment and model version behind each alias. We will evaluate the first model, then compare it with the others using the same tests.

In [lab.yaml](../data/lab.yaml), `prompt_count: 8` creates eight single-message tests and `scenario_count: 4` creates four conversations. Each conversation allows up to `max_turns: 3` user/assistant exchanges. ASSERT also generates two editable starter tests for your current policy, bringing the default total to **14 cases per target**.

Keep these small defaults for your first run: generation, conversations, and judging all use paid model calls. If you edit the policy or settings, rerun **Define the policy** to reload them, then run this cell to check the selected roles and targets.

Once these choices look right, continue below to generate tests. Neither loading the policy nor displaying these settings calls the models.

In [ ]:
from dataclasses import asdict
from app.assert_lab import provider_model
model_by_alias = {model.alias: model for model in metadata.deployments}
role_rows = [
    {'role': role, **model_by_alias[alias].to_dict(), 'provider_route': provider_model(model_by_alias[alias])}
    for role, alias in [('generator', settings.generation_target), ('tester', settings.tester_target), ('judge', settings.judge_target)]
]
display(pd.DataFrame(role_rows))
display(pd.DataFrame([model_by_alias[alias].to_dict() for alias in settings.comparison_targets]))
print(json.dumps(asdict(settings), indent=2))
print('Native configs and collections: outputs/generated/configs and outputs/generated/collections')
print('Review these settings, then continue with Generate and review tests below.')

## Generate and review tests

### Generate the taxonomy

First, ASSERT groups the behavior described in your policy into categories. It calls this a **taxonomy**. In this lab there are two categories: help the assistant may give, and behavior it must avoid.

Run this **paid generation step** to draft those categories. For our refund assistant, they should distinguish useful refund guidance from unsupported claims that a refund has happened. Each run starts a new candidate test suite.

In [ ]:
suite = create_suite(settings, metadata, root=demo_root)
print(f"Viewer setting: $env:ARTIFACTS_ROOT = '{suite.parent}'")
selection_path.write_text(json.dumps({'suite': str(suite)}, indent=2) + '\n', encoding='utf-8')
taxonomy_config = generation_config(suite, settings, metadata, taxonomy_only=True, root=demo_root)
print(f'Native taxonomy-only config: {taxonomy_config}')
display(FileLink((Path('..') / taxonomy_config.relative_to(demo_root)).as_posix()))
run_native(taxonomy_config, root=demo_root)

### Review the taxonomy

Do the categories capture the distinction you intended? Explaining refund eligibility should be permitted; inventing a refund confirmation should be prohibited.

Run the inspection cell, then open the taxonomy using the link below to read the full definitions and examples. The table is just a preview and may truncate long text. Check that there is one permitted and one prohibited category, with the right labels. If needed, edit the linked file and rerun the inspection. When you are satisfied, set `TAXONOMY_REVIEWED = True` in its own cell below and run it. Generating cases remains blocked until you confirm the review.

In [ ]:
taxonomy_path = suite / 'taxonomy.json'
taxonomy = json.loads(taxonomy_path.read_text(encoding='utf-8'))
display(FileLink((Path('..') / taxonomy_path.relative_to(demo_root)).as_posix()))
display(pd.DataFrame(taxonomy['behavior_categories']))
labels = validate_taxonomy(taxonomy)

In [ ]:
TAXONOMY_REVIEWED = True

### Generate candidate cases

Now ASSERT uses those categories to write tests. The default **paid generation step** produces:

- **8 prompts:** individual requests, split between permitted and prohibited behavior.
- **4 scenarios:** conversation goals, also split between the two categories. The tester will choose follow-up messages during evaluation, based on the assistant's replies.

The generation cell shows a table preview and a link to the complete test set. Use the link to read every request and scenario in full; clicking it does not rerun generation. These are proposed tests; the target assistants have not answered them yet.

In [ ]:
assert TAXONOMY_REVIEWED, 'Review the taxonomy first.'
generate_candidates(suite, settings, metadata, root=demo_root)
generated = read_cases(suite / 'generated-test-set.jsonl')
display(pd.DataFrame([
    {'id': case['test_case_id'], 'type': case['type'], 'category': case_category(case),
     'permitted': labels[case_category(case)], **case['seed']}
    for case in generated
]))
display(FileLink((Path('..') / (suite / 'generated-test-set.jsonl').relative_to(demo_root)).as_posix()))

### Add your own tests

Alongside ASSERT's generated tests, you may want to add specific situations you care about. Here we generate two examples to show what manually added tests look like: one testing permitted help and one testing prohibited behavior.

Run the cell below to see the examples in a table, then open `extra-tests.jsonl` using the link alongside it. Edit the examples or add your own. For instance, what might a real customer ask that the generated tests haven't covered?

To edit an example, change `seed.title` and `seed.description`. To add another test, copy an entry onto a new line and give it a unique `manual-` ID. Keep `behavior` equal to the policy name and `dimensions.behavior` equal to an exact category name from your reviewed taxonomy.

Use `type: "prompt"` for a single request or `type: "scenario"` for a conversation goal. Keep examples synthetic and do not add tools or system instructions.

The first run uses paid model calls. Rerun this cell to see your saved examples and edits again without regenerating them or making paid calls. Start a new suite when you want fresh examples or change the taxonomy.

In [ ]:
assert TAXONOMY_REVIEWED, 'Review the taxonomy first.'
generate_starter_tests(suite, settings, metadata, root=demo_root)
extra_cases = read_cases(suite / settings.extra_tests_path)
display(pd.DataFrame([
    {'id': case['test_case_id'], 'type': case['type'], 'category': case_category(case),
     'permitted': labels[case_category(case)], **case['seed']}
    for case in extra_cases
]))
display(FileLink((Path('..') / (suite / settings.extra_tests_path).relative_to(demo_root)).as_posix()))

### Review and combine tests

Would these tests tell you whether the assistant is both honest and helpful? Read every request and scenario goal. Check that each belongs in its permitted or prohibited category, and that scenarios really probe the policy rather than unrelated behavior.

Once you have reviewed both the main tests and your extra tests, set `GENERATED_CASES_REVIEWED = True` in its own cell below and run it. Then run the following cell to combine them. It also removes generated scenario-specific system instructions in a separate reviewed copy, so every target follows the same policy.

The table previews the combined set: **14 cases with the defaults**, or more if you added tests. Use the links below it to open `reviewed-test-set.jsonl` (the reviewed main tests) and `test_set.jsonl` (the combined set). Check the complete set before running the evaluation. If you edit tests, repeat this review and combine step. Reopening the links does not repeat the merge.

In [ ]:
GENERATED_CASES_REVIEWED = True

In [ ]:
reviewed_cases = review_candidates(suite, reviewed=GENERATED_CASES_REVIEWED)
all_cases = merge_cases(suite, settings, root=demo_root)
display(pd.DataFrame([
    {'id': case['test_case_id'], 'type': case['type'], 'category': case_category(case),
     'permitted': labels[case_category(case)], **case['seed']}
    for case in all_cases
]))
print(f'{len(all_cases)} candidate cases; target inference has not run.')

In [ ]:
for filename in ('reviewed-test-set.jsonl', 'test_set.jsonl'):
    display(FileLink((Path('..') / (suite / filename).relative_to(demo_root)).as_posix()))

## Run the evaluation

Let's try the first model in `comparison_targets`. These are the modesl we want to test in [lab.yaml](../data/lab.yaml)

Starting this cell freezes the inputs from **Review and combine tests**, so every compared model receives the same questions and settings. There is no separate approval step. If anything changed since review, return to that step before continuing.

ASSERT's command-line pipeline asks the questions, carries out the adaptive conversations, and has the judge assess the responses. Conversation goals are fixed, but follow-up messages adapt to each model's replies. Once evaluation has started, use a new suite for changes to the policy, tests, or shared settings.

**This is a paid evaluation**, including target, tester, and judge calls. Wait for it to finish, then use the links to open the saved transcript, scores, and metrics. Rerunning this cell starts another paid run, so use **Show results** to reopen the links and inspect an existing run.

In [ ]:
suite = Path(json.loads(selection_path.read_text(encoding='utf-8'))['suite'])
first_alias = settings.comparison_targets[0]
print(f'Evaluating reviewed inputs; first target: {first_alias}')
first_run = evaluate_target(suite, first_alias, settings, root=demo_root)
runs_path.write_text(json.dumps({'suite': str(suite), 'runs': [str(first_run)]}, indent=2) + '\n', encoding='utf-8')
print(f'Saved native run: {first_run}')
for filename in ('inference_set.jsonl', 'scores.jsonl', 'metrics.json'):
    display(FileLink((Path('..') / (first_run / filename).relative_to(demo_root)).as_posix()))

## Show results

Did the assistant respect its limits without becoming unhelpful? Load the saved results to look at two measures:

- **Policy violation rate:** among prohibited cases with usable judgments, how often did the assistant cross the line, such as claiming to have issued a refund?
- **Unnecessary refusal rate (`overrefusal`):** among permitted cases with usable judgments, how often did it refuse help it was allowed to give, such as explaining the refund rules?

Lower is better for both. Prompts and conversations are shown separately; each conversation counts as one case, including any earlier failure even if the assistant later corrects itself.

Check the case counts and `Complete case accounting` too. Missing or failed judgments are not passes, and `Unavailable` is not 0%. The extra accounting columns help explain gaps and can overlap; do not add them together. Use the links below to open full transcripts (`inference_set.jsonl`), judge assessments (`scores.jsonl`), and the native metrics and run manifest. This cell only reads saved files and makes no model calls.

In [ ]:
selected_runs = json.loads(runs_path.read_text(encoding='utf-8'))
saved = load_run(Path(selected_runs['runs'][0]))
display(pd.DataFrame(summarize_run(saved)).style.format(
    {'policy_violation_rate': '{:.1%}', 'overrefusal_rate': '{:.1%}'}, na_rep='Unavailable'))
print(f'Complete case accounting: {saved.complete}; saved run: {saved.path}')
print(f'Native supporting metrics: {saved.path / "metrics.json"}')
print(f'Native manifest: {saved.path / "manifest.json"}')
for filename in ('inference_set.jsonl', 'scores.jsonl', 'metrics.json', 'manifest.json'):
    display(FileLink((Path('..') / (saved.path / filename).relative_to(demo_root)).as_posix()))

### Inspect real failure evidence

A rate tells you how often; the conversation tells you why. Read any flagged examples alongside the judge's explanation. Did the assistant actually claim a refund had happened, or was it clearly drafting a sample message?

Check the whole exchange, not just the final reply. The judge can be wrong, and in this starter experiment it is also one of the models being compared, not an independent reviewer. If no failures were recorded, there are no examples to show. A small clean run is useful evidence, not a guarantee of safe behavior.

In [ ]:
examples = failure_examples(saved)
print(f'{len(examples)} saved assessed failures; none are synthesized.')
for example in examples:
    print(f'\n{example["test_case_id"]} (native {example["native_test_case_id"]}): {example["metric"]} / {example["seed"]}')
    display(pd.DataFrame(example['messages']))
    print(f'Justification: {example["justification"]}')
    print(f'Native citations: {example["citations"]}')
    print(f'Evidence: {example["native_scores"]}; {example["native_transcripts"]}')

## Compare models

Would another model handle the same situations differently? Keep the approved suite, policy, tester, and judge fixed, and change only the target. Adaptive conversations may take different paths because each assistant replies differently.

Run this preview to see which selected models already have matching completed results and which still need evaluation. **The preview makes no model calls.** Do not regenerate tests between targets: using the same experiment is what makes the comparison meaningful.

The final table is a side-by-side description of this small experiment, not a statistically proven ranking or an automatic pass/fail decision.

In [ ]:
selected_runs = json.loads(runs_path.read_text(encoding='utf-8'))
suite = Path(selected_runs['suite'])
approval = json.loads((suite / 'approval.json').read_text(encoding='utf-8'))
print(f'Reusing {len(approval["expected_ids"])} cases from {suite / "approval.json"}')
display(FileLink((Path('..') / (suite / 'approval.json').relative_to(demo_root)).as_posix()))
existing_runs = [load_run(Path(path)) for path in selected_runs['runs']]
remaining_targets = remaining_comparison_targets(suite, settings, metadata, existing_runs, root=demo_root)
print(f'Retaining {len(existing_runs)} completed runs; remaining targets: {remaining_targets}')

### Evaluate missing targets (paid)

Check the remaining targets in the preview. Run the next cell only when you are ready to pay for those evaluations; it reuses matching completed runs and makes no model calls if all selected targets are already complete.

Then run the display cell below it to compare the saved results. Look at both rates and their case counts, and inspect examples before drawing conclusions about a model.

In [ ]:
selected_runs = json.loads(runs_path.read_text(encoding='utf-8'))
suite = Path(selected_runs['suite'])
existing_runs = [load_run(Path(path)) for path in selected_runs['runs']]
for alias in remaining_comparison_targets(suite, settings, metadata, existing_runs, root=demo_root):
    next_run = evaluate_target(suite, alias, settings, root=demo_root)
    selected_runs['runs'].append(str(next_run))
    runs_path.write_text(json.dumps(selected_runs, indent=2) + '\n', encoding='utf-8')

In [ ]:
selected_runs = json.loads(runs_path.read_text(encoding='utf-8'))
comparison_runs = [load_run(Path(path)) for path in selected_runs['runs']]
comparison_runs.sort(key=lambda result: settings.comparison_targets.index(ModelDeployment.from_dict(result.provenance['target']).alias))
display(pd.DataFrame(compare_runs(comparison_runs)).style.format(
    {'policy_violation_rate': '{:.1%}', 'overrefusal_rate': '{:.1%}'}, na_rep='Unavailable'))
for result in comparison_runs:
    print(f'{result.provenance["run_id"]}: {result.path}; {len(failure_examples(result))} assessed failures')
    for filename in ('inference_set.jsonl', 'scores.jsonl', 'metrics.json'):
        display(FileLink((Path('..') / (result.path / filename).relative_to(demo_root)).as_posix()))

## Repeat or clean up

### Create your own policy

Ready to try your own use case? Open [04_your_own_policy.ipynb](./04_your_own_policy.ipynb). It follows the same flow, with space to write your policy and generate tests for it. Reuse the deployed models; no repeated Azure setup is needed.

### Try another model

To try another already-deployed target on the same suite, change only `comparison_targets`, reload the lab, and use **Compare models**. Keep the tester and judge fixed. A model that is not deployed needs an infrastructure change first.

### Clean up

Finished with the Azure resources? Open [X_destroy.ipynb](./X_destroy.ipynb), verify the subscription and resource group, and review every deletion confirmation before running it. Some confirmation flags may already be `True`; cleanup is destructive.